# ⚙️ Configuração Inicial

**IMPORTANTE**: Execute esta célula primeiro para configurar o ambiente do notebook.

In [1]:
# Configuração do ambiente para importação de módulos
import sys
from pathlib import Path

# Adicionar diretório raiz do projeto ao Python path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ Python path configurado: {project_root}")
print(f"✅ Diretório de trabalho: {Path().resolve()}")

✅ Python path configurado: /home/nicksson/Git/sidi/FastCheckAI
✅ Diretório de trabalho: /home/nicksson/Git/sidi/FastCheckAI/notebooks


# FastCheckAI - Pipeline de Comparação de PDFs

Este notebook implementa o pipeline completo de comparação de documentos técnicos em PDF usando processamento híbrido e análise semântica.

## Setup e Validação

Antes de começar, vamos validar que todas as dependências estão instaladas e o ambiente está configurado corretamente.

### 1.1 Validação de Imports

In [2]:
# Validação de imports essenciais
import sys
import pymupdf
import pdfplumber
import pandas as pd
from agno.agent import Agent
from agno.models.openai.chat import OpenAIChat
from openai import OpenAI

print("✅ Todos os imports essenciais foram bem-sucedidos!")
print(f"Python version: {sys.version}")
print(f"PyMuPDF version: {pymupdf.__version__}")
print(f"pdfplumber version: {pdfplumber.__version__}")
print(f"pandas version: {pd.__version__}")

# Verificar versão do agno
import agno
print(f"agno version: {agno.__version__}")

✅ Todos os imports essenciais foram bem-sucedidos!
Python version: 3.12.11 (main, Aug 28 2025, 17:07:59) [Clang 20.1.4 ]
PyMuPDF version: 1.26.4
pdfplumber version: 0.11.7
pandas version: 2.3.3
agno version: 2.0.11


### 1.2 Validação de Configuração (API Keys)

In [3]:
# Validação de configuração e API keys
from src.config import (
    OPENAI_API_KEY,
    MODEL,
    TEMPERATURE,
    FUZZY_MATCH_THRESHOLD,
    SEMANTIC_CONFIDENCE_THRESHOLD,
    logger
)

print("✅ Configuração carregada com sucesso!")
print(f"Modelo: {MODEL}")
print(f"Temperature: {TEMPERATURE}")
print(f"Fuzzy Match Threshold: {FUZZY_MATCH_THRESHOLD}")
print(f"Semantic Confidence Threshold: {SEMANTIC_CONFIDENCE_THRESHOLD}")
print(f"API Key configurada: {'Sim' if OPENAI_API_KEY else 'Não'}")

2025-10-02 17:37:23 | INFO     | src.config | Configuration loaded successfully
2025-10-02 17:37:23 | INFO     | src.config | Model: gpt-4o | Temperature: 0.3
2025-10-02 17:37:23 | INFO     | src.config | Fuzzy Match Threshold: 0.8


✅ Configuração carregada com sucesso!
Modelo: gpt-4o
Temperature: 0.3
Fuzzy Match Threshold: 0.8
Semantic Confidence Threshold: 0.7
API Key configurada: Sim


### 1.3 Teste de Conexão OpenAI API

In [4]:
# Teste de conexão com OpenAI API
from openai import OpenAI
import time

client = OpenAI(api_key=OPENAI_API_KEY)

start_time = time.time()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Fale Olá em uma palavra"}],
    temperature=TEMPERATURE
)
elapsed_time = time.time() - start_time

print(f"✅ Conexão OpenAI API bem-sucedida!")
print(f"Resposta: {response.choices[0].message.content}")
print(f"Tempo de resposta: {elapsed_time:.2f}s")
print(f"Tokens usados: {response.usage.total_tokens}")

2025-10-02 17:37:24 | INFO     | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Conexão OpenAI API bem-sucedida!
Resposta: Oi!
Tempo de resposta: 0.98s
Tokens usados: 16


### 1.4 Validação Agno Framework

O Agno Framework será usado para orquestração de agentes LLM para análise semântica. Vamos validar que funciona corretamente.

**Notas importantes:**
- Importações corretas: 
  - `from agno.agent import Agent`
  - `from agno.models.openai.chat import OpenAIChat`
- O Agent requer um objeto Model, não uma string
- Criar modelo com: `OpenAIChat(id="gpt-4o")`

In [5]:
# Teste Agno Framework - Hello World
from agno.agent import Agent
from agno.models.openai.chat import OpenAIChat
import time

# Criar modelo OpenAI
model = OpenAIChat(id=MODEL)

# Criar agente Agno com GPT-4o
agent = Agent(
    model=model,
    instructions="You are a helpful assistant specialized in technical documentation.",
)

# Teste simples
start_time = time.time()
response = agent.run("Explain what a technical standard is in one sentence")
elapsed_time = time.time() - start_time

print("✅ Agno Framework validado com sucesso!")
print(f"Resposta do agente: {response.content}")
print(f"Tempo de resposta: {elapsed_time:.2f}s")

2025-10-02 17:37:26 | INFO     | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-02 17:37:27 | INFO     | httpx | HTTP Request: POST https://os-api.agno.com/telemetry/runs "HTTP/1.1 201 Created"


✅ Agno Framework validado com sucesso!
Resposta do agente: A technical standard is a document that establishes uniform engineering or technical criteria, methods, processes, or practices consistently applied to ensure quality and interoperability.
Tempo de resposta: 2.42s


---

## ✅ Setup Completo!

Se todas as células acima foram executadas sem erros, seu ambiente está pronto para desenvolvimento.

**Próximos passos:**
1. Implementar pipeline de extração de PDF (Feature 2)
2. Desenvolver sistema de alinhamento heurístico (Feature 3)
3. Integrar análise semântica com Agno (Feature 4)

---

## 2. Carregamento e Validação de PDFs

Esta seção implementa o carregamento robusto dos dois PDFs a serem comparados com:
- Validação de tamanho (≤25MB)
- Detecção automática de tipo (nativo vs escaneado)
- Tratamento de erros claros

### 2.1 Carregamento dos PDFs

In [6]:
PDF_PATH_01 = '../data/ASTM_A29_A29M_Rev.00\'2015.pdf'
PDF_PATH_02 = '../data/ASTM_A29_A29M_Rev.00\'2016.pdf'

In [7]:
# Importar módulo de carregamento de PDFs
from src.pdf_loader import load_pdf, detect_pdf_type

# Carregar os dois PDFs para comparação
print("📄 Carregando PDFs...")
pdf_a = load_pdf(PDF_PATH_01)
pdf_b = load_pdf(PDF_PATH_02)

print(f"\n✅ PDF A carregado: {pdf_a.page_count} páginas")
print(f"✅ PDF B carregado: {pdf_b.page_count} páginas")

2025-10-02 17:37:27 | INFO     | src.pdf_loader | Validating PDF size: ASTM_A29_A29M_Rev.00'2015.pdf (0.2MB)
2025-10-02 17:37:27 | INFO     | src.pdf_loader | PDF carregado: ASTM_A29_A29M_Rev.00'2015.pdf (17 páginas, 0.2MB)
2025-10-02 17:37:27 | INFO     | src.pdf_loader | Validating PDF size: ASTM_A29_A29M_Rev.00'2016.pdf (0.2MB)
2025-10-02 17:37:27 | INFO     | src.pdf_loader | PDF carregado: ASTM_A29_A29M_Rev.00'2016.pdf (17 páginas, 0.2MB)


📄 Carregando PDFs...

✅ PDF A carregado: 17 páginas
✅ PDF B carregado: 17 páginas


### 2.2 Detecção de Tipo de PDF

In [8]:
# Detectar tipo dos PDFs (nativo vs escaneado)
print("🔍 Detectando tipo dos PDFs...\n")

type_a = detect_pdf_type(pdf_a)
type_b = detect_pdf_type(pdf_b)

print(f"PDF A: {'✅ Nativo (texto extraível)' if type_a['is_native'] else '⚠️  Escaneado (requer OCR)'}")
print(f"  └─ {type_a['char_count']} caracteres na página 1")

print(f"\nPDF B: {'✅ Nativo (texto extraível)' if type_b['is_native'] else '⚠️  Escaneado (requer OCR)'}")
print(f"  └─ {type_b['char_count']} caracteres na página 1")

# Avisar se algum PDF requer OCR
if type_a['requires_ocr'] or type_b['requires_ocr']:
    print("\n⚠️  ATENÇÃO: Um ou mais PDFs parecem ser escaneados e podem requerer OCR para extração completa.")

2025-10-02 17:37:27 | INFO     | src.pdf_loader | PDF type detected: Nativo (extração direta) (1445 characters on page 1)
2025-10-02 17:37:27 | INFO     | src.pdf_loader | PDF type detected: Nativo (extração direta) (5596 characters on page 1)


🔍 Detectando tipo dos PDFs...

PDF A: ✅ Nativo (texto extraível)
  └─ 1445 caracteres na página 1

PDF B: ✅ Nativo (texto extraível)
  └─ 5596 caracteres na página 1


---

## 3. Extração de Texto e Parsing de Estrutura

Esta seção implementa a extração completa de texto preservando hierarquia de seções e metadados.

### 3.1 Extração de Metadados dos PDFs

In [9]:
# Extrair metadados dos PDFs
from src.text_extractor import extract_metadata

print("📊 Extraindo metadados dos PDFs...\n")

metadata_a = extract_metadata(pdf_a)
metadata_b = extract_metadata(pdf_b)

print("PDF A:")
for key, value in metadata_a.items():
    print(f"  {key}: {value}")

print("\nPDF B:")
for key, value in metadata_b.items():
    print(f"  {key}: {value}")

2025-10-02 17:37:27 | INFO     | src.text_extractor | Metadata extracted: 17 pages, title: N/A
2025-10-02 17:37:27 | INFO     | src.text_extractor | Metadata extracted: 17 pages, title: N/A


📊 Extraindo metadados dos PDFs...

PDF A:
  page_count: 17
  format: PDF 1.7
  title: N/A
  author: N/A
  subject: N/A
  keywords: N/A
  creator: N/A
  producer: PDF-XChange Printer 2012 (5.0 build 265) [Windows 7 Enterprise Professional (Build 7601: Service Pack 1)]
  creation_date: 2016-01-11

PDF B:
  page_count: 17
  format: PDF 1.6
  title: N/A
  author: N/A
  subject: N/A
  keywords: N/A
  creator: XPP
  producer: PDPreStamp v3.3
  creation_date: 2017-02-23


### 3.2 Extração do texto completo

In [10]:
# Extrair texto completo dos PDFs com marcadores de página
from src.text_extractor import extract_text
import time

print("📄 Extraindo texto dos PDFs...\n")

# PDF A
start_time = time.time()
text_a = extract_text(pdf_a, include_page_markers=True)
time_a = time.time() - start_time

print(f"✅ PDF A: {len(text_a)} caracteres extraídos em {time_a:.2f}s")
print(f"   Taxa: {len(text_a)/time_a:.0f} chars/s")
print(f"   Primeiros 200 caracteres: {text_a[:200]}...")

# PDF B
print()
start_time = time.time()
text_b = extract_text(pdf_b, include_page_markers=True)
time_b = time.time() - start_time

print(f"✅ PDF B: {len(text_b)} caracteres extraídos em {time_b:.2f}s")
print(f"   Taxa: {len(text_b)/time_b:.0f} chars/s")
print(f"   Primeiros 200 caracteres: {text_b[:200]}...")

2025-10-02 17:37:27 | INFO     | src.text_extractor | Starting hybrid text extraction from 17 pages (OCR: enabled)
2025-10-02 17:37:27 | INFO     | src.text_extractor | Page 2: Corrupted text detected, using OCR fallback


📄 Extraindo texto dos PDFs...



2025-10-02 17:37:29 | INFO     | src.text_extractor | Page 3: Corrupted text detected, using OCR fallback
2025-10-02 17:37:32 | INFO     | src.text_extractor | Page 4: Corrupted text detected, using OCR fallback
2025-10-02 17:37:34 | INFO     | src.text_extractor | Page 5: Corrupted text detected, using OCR fallback
2025-10-02 17:37:36 | INFO     | src.text_extractor | Page 6: Corrupted text detected, using OCR fallback
2025-10-02 17:37:38 | INFO     | src.text_extractor | Page 7: Corrupted text detected, using OCR fallback
2025-10-02 17:37:41 | INFO     | src.text_extractor | Page 8: Corrupted text detected, using OCR fallback
2025-10-02 17:37:43 | INFO     | src.text_extractor | Page 9: Corrupted text detected, using OCR fallback
2025-10-02 17:37:44 | INFO     | src.text_extractor | Page 10: Corrupted text detected, using OCR fallback
2025-10-02 17:37:47 | INFO     | src.text_extractor | Extracted page 10/17
2025-10-02 17:37:47 | INFO     | src.text_extractor | Page 11: Corrupted tex

✅ PDF A: 65498 caracteres extraídos em 31.51s
   Taxa: 2079 chars/s
   Primeiros 200 caracteres: 
--- PAGE 1 ---
COPYRIGHT NOTICE
This document is copyrighted by the American Society for Testing
and Materials (ASTM), 100 Barr Harbor Drive, West Conshohocken,
PA 19428-2959, USA.  All rights reserv...

✅ PDF B: 68444 caracteres extraídos em 0.03s
   Taxa: 2055056 chars/s
   Primeiros 200 caracteres: 
--- PAGE 1 ---
Designation: A29/A29M −16
Standard Speciﬁcation for
General Requirements for Steel Bars, Carbon and Alloy,
Hot-Wrought1
This standard is issued under the ﬁxed designation A29/A29M; the...


### 3.3 Parsing de Hierarquia de Seções

In [11]:
# Fazer parsing da hierarquia de seções
from src.text_extractor import parse_section_hierarchy

print("🔍 Fazendo parsing da hierarquia de seções...\n")

# PDF A
sections_a = parse_section_hierarchy(text_a)
print(f"PDF A: {len(sections_a)} seções de nível 1")
print(f"Seções encontradas: {list(sections_a.keys())[:10]}")  # Primeiras 10

# Mostrar exemplo de estrutura hierárquica
if sections_a:
    first_section_id = list(sections_a.keys())[0]
    first_section = sections_a[first_section_id]
    print(f"\nExemplo de seção (ID: {first_section_id}):")
    print(f"  Título: {first_section['title']}")
    print(f"  Nível: {first_section['level']}")
    print(f"  Conteúdo (primeiros 150 chars): {first_section['content'][:150]}...")
    print(f"  Subseções: {list(first_section['subsections'].keys())}")

print("\n" + "="*60 + "\n")

# PDF B
sections_b = parse_section_hierarchy(text_b)
print(f"PDF B: {len(sections_b)} seções de nível 1")
print(f"Seções encontradas: {list(sections_b.keys())[:10]}")  # Primeiras 10

# Mostrar exemplo de estrutura hierárquica
if sections_b:
    first_section_id = list(sections_b.keys())[0]
    first_section = sections_b[first_section_id]
    print(f"\nExemplo de seção (ID: {first_section_id}):")
    print(f"  Título: {first_section['title']}")
    print(f"  Nível: {first_section['level']}")
    print(f"  Conteúdo (primeiros 150 chars): {first_section['content'][:150]}...")
    print(f"  Subseções: {list(first_section['subsections'].keys())}")

2025-10-02 17:37:58 | INFO     | src.text_extractor | Found 85 section headers
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.2.3 at level 3
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.2.4 at level 3
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.3.3 at level 3
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.3.4 at level 3
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.3.5 at level 3
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.3.6 at level 3
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 4.3.7 at level 3
2025-10-02 17:37:58 | INFO     | src.text_extractor | Section hierarchy parsed: {1: 23, 2: 29, 3: 25, 4: 8}
2025-10-02 17:37:58 | INFO     | src.text_extractor | Found 237 section headers
2025-10-02 17:37:58 | WARNING  | src.text_extractor | Orphaned section 1.1 at level 2
2025-10-02 17:37:58 | WARNING  

🔍 Fazendo parsing da hierarquia de seções...

PDF A: 19 seções de nível 1
Seções encontradas: ['7', '3', '4', '4.2.3', '4.2.4', '5', '4.3.3', '4.3.4', '4.3.5', '4.3.6']

Exemplo de seção (ID: 7):
  Título: Designation: A29/A29M - 15
  Nível: 1
  Conteúdo (primeiros 150 chars): “ull

INTERNATIONAL

Standard Specification for

General Requirements for Steel Bars, Carbon and Alloy,

Hot-Wrought'

This standard is issued under t...
  Subseções: ['1.1', '1.2', '1.3', '1.4', '2.1', '2.2', '2.3', '2.4', '3.1', '4.1', '4.2']


PDF B: 36 seções de nível 1
Seções encontradas: ['1.1', '1.2', '1.3', '1.4', '2.1', '1', '2', '3', '4', '5']

Exemplo de seção (ID: 1.1):
  Título: This speciﬁcation2 covers a group of common require-
  Nível: 2
  Conteúdo (primeiros 150 chars): ments which, unless otherwise speciﬁed in the purchase order
or in an individual speciﬁcation, shall apply to carbon and alloy
steel bars under each o...
  Subseções: []


### 3.4 Estatísticas de Extração

In [12]:
# Calcular estatísticas de extração
def count_sections_recursive(sections_dict):
    """Conta total de seções recursivamente"""
    count = len(sections_dict)
    for section in sections_dict.values():
        count += count_sections_recursive(section.get('subsections', {}))
    return count

print("📊 Estatísticas de Extração\n")
print("="*60)

print(f"\nPDF A:")
print(f"  Páginas: {metadata_a['page_count']}")
print(f"  Caracteres extraídos: {len(text_a):,}")
print(f"  Seções nível 1: {len(sections_a)}")
print(f"  Total de seções (todos níveis): {count_sections_recursive(sections_a)}")
print(f"  Tempo de extração: {time_a:.2f}s")

print(f"\nPDF B:")
print(f"  Páginas: {metadata_b['page_count']}")
print(f"  Caracteres extraídos: {len(text_b):,}")
print(f"  Seções nível 1: {len(sections_b)}")
print(f"  Total de seções (todos níveis): {count_sections_recursive(sections_b)}")
print(f"  Tempo de extração: {time_b:.2f}s")

print("\n" + "="*60)
print("✅ Extração de texto completa! Dados prontos para alinhamento e comparação.")

📊 Estatísticas de Extração


PDF A:
  Páginas: 17
  Caracteres extraídos: 65,498
  Seções nível 1: 19
  Total de seções (todos níveis): 38
  Tempo de extração: 31.51s

PDF B:
  Páginas: 17
  Caracteres extraídos: 68,444
  Seções nível 1: 36
  Total de seções (todos níveis): 117
  Tempo de extração: 0.03s

✅ Extração de texto completa! Dados prontos para alinhamento e comparação.


### 3.5 Testes de Validação

Vamos testar os casos de erro para garantir que as validações estão funcionando corretamente.

In [13]:
# Teste 1: Validação de tamanho (PDF muito grande)
print("🧪 Teste 1: Validação de tamanho máximo\n")
try:
    large_pdf = load_pdf("data/inputs/large_file.pdf")
    print("❌ FALHA: PDF grande deveria ter sido rejeitado")
except Exception as e:
    print(f"✅ SUCESSO: {type(e).__name__}: {e}\n")

# Teste 2: PDF corrompido
print("🧪 Teste 2: Tratamento de PDF corrompido\n")
try:
    corrupt_pdf = load_pdf("data/inputs/corrupted.pdf")
    print("❌ FALHA: PDF corrompido deveria ter gerado erro")
except Exception as e:
    print(f"✅ SUCESSO: {type(e).__name__}: {e}\n")

# Teste 3: Arquivo inexistente
print("🧪 Teste 3: Tratamento de arquivo inexistente\n")
try:
    missing_pdf = load_pdf("data/inputs/nao_existe.pdf")
    print("❌ FALHA: Arquivo inexistente deveria gerar erro")
except FileNotFoundError as e:
    print(f"✅ SUCESSO: FileNotFoundError: {e}\n")

# Teste 4: PDF escaneado (sem texto)
print("🧪 Teste 4: Detecção de PDF escaneado\n")
try:
    scanned_pdf = load_pdf("data/inputs/scanned_sample.pdf")
    scanned_type = detect_pdf_type(scanned_pdf)
    if scanned_type['requires_ocr']:
        print(f"✅ SUCESSO: PDF detectado como escaneado ({scanned_type['char_count']} chars)")
    else:
        print(f"❌ FALHA: PDF deveria ser detectado como escaneado")
    scanned_pdf.close()
except Exception as e:
    print(f"❌ ERRO: {e}")

print("\n✅ Todos os testes de validação concluídos!")

2025-10-02 17:37:58 | ERROR    | src.pdf_loader | PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/large_file.pdf
2025-10-02 17:37:58 | ERROR    | src.pdf_loader | PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/corrupted.pdf
2025-10-02 17:37:58 | ERROR    | src.pdf_loader | PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/nao_existe.pdf
2025-10-02 17:37:58 | ERROR    | src.pdf_loader | PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/scanned_sample.pdf


🧪 Teste 1: Validação de tamanho máximo

✅ SUCESSO: FileNotFoundError: PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/large_file.pdf

🧪 Teste 2: Tratamento de PDF corrompido

✅ SUCESSO: FileNotFoundError: PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/corrupted.pdf

🧪 Teste 3: Tratamento de arquivo inexistente

✅ SUCESSO: FileNotFoundError: PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/nao_existe.pdf

🧪 Teste 4: Detecção de PDF escaneado

❌ ERRO: PDF file not found: /home/nicksson/Git/sidi/FastCheckAI/notebooks/data/inputs/scanned_sample.pdf

✅ Todos os testes de validação concluídos!


---

## 4. Detecção e Extração de Tabelas

Esta seção implementa a extração de tabelas dos PDFs usando:
- PyMuPDF para detecção rápida de páginas com tabelas
- pdfplumber para extração precisa de dados tabulares
- Metadados ricos para posterior alinhamento entre documentos

### 4.1 Detecção de Páginas com Tabelas

In [14]:
# Detectar páginas que contêm tabelas
from src.table_extractor import detect_table_pages
import time

print("🔍 Detectando páginas com tabelas...\n")

# PDF A
start_time = time.time()
table_pages_a = detect_table_pages(pdf_a)
time_a = time.time() - start_time

print(f"✅ PDF A: {len(table_pages_a)} páginas com tabelas detectadas em {time_a:.2f}s")
print(f"   Páginas: {table_pages_a}")

# PDF B
print()
start_time = time.time()
table_pages_b = detect_table_pages(pdf_b)
time_b = time.time() - start_time

print(f"✅ PDF B: {len(table_pages_b)} páginas com tabelas detectadas em {time_b:.2f}s")
print(f"   Páginas: {table_pages_b}")

2025-10-02 17:37:58 | INFO     | src.table_extractor | Starting hybrid table detection on 17 pages (line-based: H≥10, V≥5, text-based: enabled)
2025-10-02 17:37:59 | INFO     | src.table_extractor | Hybrid table detection complete: 2 pages with tables in 0.05s
2025-10-02 17:37:59 | INFO     | src.table_extractor | Starting hybrid table detection on 17 pages (line-based: H≥10, V≥5, text-based: enabled)
2025-10-02 17:37:59 | INFO     | src.table_extractor | Hybrid table detection complete: 8 pages with tables in 0.04s


🔍 Detectando páginas com tabelas...

✅ PDF A: 2 páginas com tabelas detectadas em 0.05s
   Páginas: [3, 4]

✅ PDF B: 8 páginas com tabelas detectadas em 0.04s
   Páginas: [1, 2, 3, 4, 5, 6, 7, 8]


### 4.2 Extração de Tabelas

In [15]:
# Extrair tabelas das páginas detectadas
from src.table_extractor import extract_tables

print("📊 Extraindo tabelas dos PDFs...\n")

# PDF A
if table_pages_a:
    result_a = extract_tables(PDF_PATH_01, table_pages_a)
    print(result_a.summary())
    print(f"\n  Tabelas extraídas: {result_a.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_a.failed_pages)}")
else:
    print("PDF A: Nenhuma página com tabelas detectada")
    result_a = None

print("\n" + "="*60 + "\n")

# PDF B
if table_pages_b:
    result_b = extract_tables(PDF_PATH_02, table_pages_b)
    print(result_b.summary())
    print(f"\n  Tabelas extraídas: {result_b.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_b.failed_pages)}")
else:
    print("PDF B: Nenhuma página com tabelas detectada")
    result_b = None

2025-10-02 17:37:59 | INFO     | src.table_extractor | Starting hybrid table extraction from 2 pages (default + text-based strategies)


📊 Extraindo tabelas dos PDFs...



2025-10-02 17:37:59 | INFO     | src.table_extractor | Table extraction complete: 2 tables from 2 pages in 0.32s
2025-10-02 17:37:59 | INFO     | src.table_extractor | Starting hybrid table extraction from 8 pages (default + text-based strategies)


Table Extraction Summary:
  Pages with tables: 2
  Tables extracted: 2
  Failed pages: 0
  Detection time: 0.00s
  Extraction time: 0.32s
  Total time: 0.32s

  Tabelas extraídas: 2
  Páginas com falha: 0




2025-10-02 17:38:00 | INFO     | src.table_extractor | Table extraction complete: 8 tables from 8 pages in 1.35s


Table Extraction Summary:
  Pages with tables: 8
  Tables extracted: 8
  Failed pages: 0
  Detection time: 0.00s
  Extraction time: 1.35s
  Total time: 1.35s

  Tabelas extraídas: 8
  Páginas com falha: 0


### 4.3 Visualização das Tabelas Extraídas

In [16]:
# Mostrar as primeiras tabelas extraídas
print("📋 Visualizando tabelas extraídas...\n")

# PDF A - Mostrar primeiras 3 tabelas
if result_a and result_a.tables:
    print(f"PDF A - Primeiras {min(3, len(result_a.tables))} tabelas:\n")
    
    for i, table in enumerate(result_a.tables[:3]):
        print(f"Tabela {i+1} (Página {table.metadata.page_num + 1}):")
        print(f"  Dimensão: {table.metadata.row_count}x{table.metadata.column_count}")
        print(f"  Header: {'Sim' if table.metadata.has_header else 'Não'}")
        print(f"  Confiança: {table.metadata.confidence:.2f}")
        print(f"  Seção: {table.metadata.section_context or 'N/A'}")
        print("\nPrimeiras linhas:")
        display(table.data.head(5))
        print("\n" + "-"*60 + "\n")
else:
    print("PDF A: Nenhuma tabela extraída\n")

print("="*60 + "\n")

# PDF B - Mostrar primeiras 3 tabelas
if result_b and result_b.tables:
    print(f"PDF B - Primeiras {min(3, len(result_b.tables))} tabelas:\n")
    
    for i, table in enumerate(result_b.tables[:3]):
        print(f"Tabela {i+1} (Página {table.metadata.page_num + 1}):")
        print(f"  Dimensão: {table.metadata.row_count}x{table.metadata.column_count}")
        print(f"  Header: {'Sim' if table.metadata.has_header else 'Não'}")
        print(f"  Confiança: {table.metadata.confidence:.2f}")
        print(f"  Seção: {table.metadata.section_context or 'N/A'}")
        print("\nPrimeiras linhas:")
        display(table.data.head(5))
        print("\n" + "-"*60 + "\n")
else:
    print("PDF B: Nenhuma tabela extraída")

📋 Visualizando tabelas extraídas...

PDF A - Primeiras 2 tabelas:

Tabela 1 (Página 4):
  Dimensão: 83x4
  Header: Sim
  Confiança: 0.85
  Seção: N/A

Primeiras linhas:


,,ÌßÞÔÛïÙ®¿¼»Ü»­·¹²¿¬·±²­¿²¼Ý¸»³·½¿´Ý±³°±­·¬·±²­±ºÝ¿®¾±²Í¬»»´Þ¿®­,,
0,,,,
1,,Ø»¿¬Ý¸»³·½¿´Î¿²¹»­¿²¼Ô·³·¬­ôû,,
2,Ù®¿¼»Ü»­·¹²¿¬·±²,,,
3,,Ý¿®¾±² Ó¿²¹¿²»­» Ð¸±­°¸±®«­ô³¿¨,Í«´º«®ô³,¿¨ß
4,,,,



------------------------------------------------------------

Tabela 2 (Página 5):
  Dimensão: 74x1
  Header: Sim
  Confiança: 0.42
  Seção: N/A

Primeiras linhas:


,
0,Ù®¿¼»Ü»­·¹²¿¬·±² Ý¿®¾±² Ó¿²¹¿²»­» Ð¸±­°¸±®±«­ ...
1,
2,ïîïí ðòïí³¿¨ ðòéðPïòðð ðòðéPðòïî ðòîìPðòíí òòò
3,ïîïë ðòðç³¿¨ ðòéëPïòðë ðòðìPðòðç ðòîêPðòíë òòò
4,ïîÔïí ðòïí³¿¨ ðòéðPïòðð ðòðéPðòïî ðòîìPðòíí ðò...



------------------------------------------------------------


PDF B - Primeiras 3 tabelas:

Tabela 1 (Página 2):
  Dimensão: 77x3
  Header: Sim
  Confiança: 0.77
  Seção: N/A

Primeiras linhas:


...


------------------------------------------------------------

Tabela 2 (Página 3):
  Dimensão: 84x9
  Header: Sim
  Confiança: 0.81
  Seção: N/A

Primeiras linhas:


,,,TABLE1Grade,Designationsand,ChemicalCompositi,onsofCarbonSteelB,ars,,
0,,,,,,,,,
1,,,,,HeatChemicalRan,"gesandLimits,%",,,
2,,GradeDesignation,,,,,,,
3,,,Carbon,,Manganese,"Phosphorus,max",,"Sulfur,m",axA
4,,,,,,,,,



------------------------------------------------------------

Tabela 3 (Página 4):
  Dimensão: 79x3
  Header: Sim
  Confiança: 0.78
  Seção: N/A

Primeiras linhas:


,,,
0,,Grade,Designation Carbon Manganese Phosphorous Sulfu...
1,,,
2,,1213,0.13max 0.70–1.00 0.07–0.12 0.24–0.33 ...
3,,1215,0.09max 0.75–1.05 0.04–0.09 0.26–0.35 ...
4,,12L13,0.13max 0.70–1.00 0.07–0.12 0.24–0.33 0.15–0.35



------------------------------------------------------------



### 4.4 Estatísticas de Extração de Tabelas

In [17]:
# Calcular estatísticas de extração de tabelas
print("📊 Estatísticas de Extração de Tabelas\n")
print("="*60)

if result_a:
    print(f"\nPDF A:")
    print(f"  Páginas com tabelas: {result_a.total_pages_detected}")
    print(f"  Tabelas extraídas: {result_a.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_a.failed_pages)}")
    print(f"  Tempo de detecção: {result_a.detection_time_seconds:.2f}s")
    print(f"  Tempo de extração: {result_a.extraction_time_seconds:.2f}s")
    print(f"  Tempo total: {result_a.detection_time_seconds + result_a.extraction_time_seconds:.2f}s")
    
    if result_a.tables:
        avg_confidence = sum(t.metadata.confidence for t in result_a.tables) / len(result_a.tables)
        print(f"  Confiança média: {avg_confidence:.2f}")
else:
    print(f"\nPDF A: Nenhuma tabela detectada")

if result_b:
    print(f"\nPDF B:")
    print(f"  Páginas com tabelas: {result_b.total_pages_detected}")
    print(f"  Tabelas extraídas: {result_b.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_b.failed_pages)}")
    print(f"  Tempo de detecção: {result_b.detection_time_seconds:.2f}s")
    print(f"  Tempo de extração: {result_b.extraction_time_seconds:.2f}s")
    print(f"  Tempo total: {result_b.detection_time_seconds + result_b.extraction_time_seconds:.2f}s")
    
    if result_b.tables:
        avg_confidence = sum(t.metadata.confidence for t in result_b.tables) / len(result_b.tables)
        print(f"  Confiança média: {avg_confidence:.2f}")
else:
    print(f"\nPDF B: Nenhuma tabela detectada")

print("\n" + "="*60)
print("✅ Extração de tabelas completa! Dados prontos para comparação.")

📊 Estatísticas de Extração de Tabelas


PDF A:
  Páginas com tabelas: 2
  Tabelas extraídas: 2
  Páginas com falha: 0
  Tempo de detecção: 0.00s
  Tempo de extração: 0.32s
  Tempo total: 0.32s
  Confiança média: 0.63

PDF B:
  Páginas com tabelas: 8
  Tabelas extraídas: 8
  Páginas com falha: 0
  Tempo de detecção: 0.00s
  Tempo de extração: 1.35s
  Tempo total: 1.35s
  Confiança média: 0.77

✅ Extração de tabelas completa! Dados prontos para comparação.


---

## 5. Alinhamento de Seções

Esta seção alinha seções correspondentes entre os dois PDFs usando estratégia hierárquica:
1. **Exact match**: Alinhamento por ID de seção idêntico
2. **Fuzzy match**: Alinhamento por similaridade de título (threshold ≥ 0.8)
3. **LLM suggestion**: Agno sugere alinhamento para casos ambíguos

### 5.1 Alinhamento Heurístico de Seções

In [ ]:
# Importar função COMPLETA de alinhamento
from src.section_aligner import align_sections
import time

print("🔗 Alinhando seções entre PDFs...")
start_time = time.time()

# Executar alinhamento COMPLETO (heuristic + detecção de unmatched)
# use_llm_fallback=False para execução rápida (sem chamadas LLM)
alignment_result = align_sections(sections_a, sections_b, use_llm_fallback=False)
elapsed = time.time() - start_time

print(f"✅ Alinhamento concluído em {elapsed:.2f}s\n")
print(f"Estatísticas:")
print(f"  Seções alinhadas: {alignment_result['metadata']['aligned_count']}")
print(f"  Seções adicionadas (B): {alignment_result['metadata']['added_count']}")
print(f"  Seções removidas (A): {alignment_result['metadata']['removed_count']}")
print(f"  Confiança média: {alignment_result['metadata']['avg_confidence']:.2f}")

# Contar tipos de alinhamento
exact_matches = sum(1 for a in alignment_result['alignments'].values() if a['method'] == 'exact')
fuzzy_matches = sum(1 for a in alignment_result['alignments'].values() if a['method'] == 'fuzzy')

print(f"\nTipos de alinhamento:")
print(f"  Exact match: {exact_matches}")
print(f"  Fuzzy match: {fuzzy_matches}")

### 4.2 Visualização de Alinhamentos

In [ ]:
# Mostrar primeiros 10 pares alinhados
print("📋 Primeiros 10 pares de seções alinhadas:\n")
print("="*80)

# Converter alignments dict para lista para facilitar iteração
alignments_list = list(alignment_result['alignments'].items())

for i, (section_id_a, alignment_data) in enumerate(alignments_list[:10], 1):
    section_id_b = alignment_data['section_b_id']
    print(f"\n{i}. {section_id_a} ↔ {section_id_b}")
    print(f"   Tipo: {alignment_data['method']} | Confiança: {alignment_data['confidence']:.2f}")
    print(f"   Título A: {alignment_data['title_a'][:60]}...")
    print(f"   Título B: {alignment_data['title_b'][:60]}...")

# Mostrar seções não correspondidas
print(f"\n{'='*80}")
print(f"\n⚠️  Seções removidas do PDF A ({len(alignment_result['removed'])}):")
for section_info in alignment_result['removed'][:5]:
    print(f"  - {section_info['id']}: {section_info['title'][:50]}...")

print(f"\n⚠️  Seções adicionadas no PDF B ({len(alignment_result['added'])}):")
for section_info in alignment_result['added'][:5]:
    print(f"  - {section_info['id']}: {section_info['title'][:50]}...")

print(f"\n{'='*80}")
print("✅ Alinhamento de seções completo! Dados prontos para comparação textual.")

---

## 6. Comparação Textual

Esta seção compara o texto de seções alinhadas usando difflib:
- Detecta adições, remoções e modificações
- Filtra diferenças triviais (whitespace-only)
- Destaca mudanças numéricas (valores com unidades)
- Prioriza termos críticos (mandatory, shall, etc.)

### 6.1 Comparação de Seções Alinhadas

In [ ]:
# Importar módulo de comparação textual
from src.text_comparator import compare_text
import time

print("📝 Comparando texto de seções alinhadas...")
print("="*80)

# Comparar primeiras 5 seções alinhadas
comparison_results = []

# Converter alignments para lista
alignments_list = list(alignment_result['alignments'].items())

for i, (section_id_a, alignment_data) in enumerate(alignments_list[:5], 1):
    section_id_b = alignment_data['section_b_id']
    section_a = sections_a[section_id_a]
    section_b = sections_b[section_id_b]
    
    print(f"\n{i}. Comparando {section_id_a} ↔ {section_id_b}")
    
    start_time = time.time()
    diff_result = compare_text(section_a['content'], section_b['content'])
    elapsed = time.time() - start_time
    
    total_diffs = sum(len(v) for v in diff_result.values())
    
    print(f"   Tempo: {elapsed:.3f}s")
    print(f"   Diferenças: {total_diffs} total")
    print(f"   - Adições: {len(diff_result['additions'])}")
    print(f"   - Remoções: {len(diff_result['removals'])}")
    print(f"   - Modificações: {len(diff_result['modifications'])}")
    
    # Contar diffs flagados
    numeric_count = sum(1 for d in diff_result['modifications'] if d.get('contains_numeric_change'))
    critical_count = sum(1 for d in diff_result['modifications'] if d.get('contains_critical_term'))
    
    if numeric_count > 0 or critical_count > 0:
        print(f"   🚨 Flags: {numeric_count} numéricos, {critical_count} críticos")
    
    comparison_results.append({
        'section_id_a': section_id_a,
        'section_id_b': section_id_b,
        'diffs': diff_result,
        'total_diffs': total_diffs,
        'time': elapsed
    })

print(f"\n{'='*80}")
print(f"✅ Comparação textual de {len(comparison_results)} seções completa!")

### 6.2 Visualização de Diferenças Detectadas

In [ ]:
# Visualizar diferenças da primeira seção comparada
if comparison_results:
    print("📋 Exemplo de diferenças detectadas (primeira seção):\n")
    print("="*80)
    
    first_result = comparison_results[0]
    diffs = first_result['diffs']
    section_id_a = first_result['section_id_a']
    section_id_b = first_result['section_id_b']
    
    # Buscar títulos das seções
    title_a = alignment_result['alignments'][section_id_a]['title_a']
    
    print(f"Seção: {section_id_a} ↔ {section_id_b}")
    print(f"Título: {title_a[:60]}...\n")
    
    # Mostrar modificações (primeiras 3)
    if diffs['modifications']:
        print(f"Modificações ({len(diffs['modifications'])} total):\n")
        for i, mod in enumerate(diffs['modifications'][:3], 1):
            flags = []
            if mod.get('contains_numeric_change'):
                flags.append('🔢 NUMÉRICO')
            if mod.get('contains_critical_term'):
                flags.append('⚠️ CRÍTICO')
            
            flag_str = ' '.join(flags) if flags else ''
            print(f"{i}. {flag_str}")
            print(f"   Original: '{mod['original'][:80]}...'")
            print(f"   Novo:     '{mod['content'][:80]}...'")
            print()
    
    # Mostrar adições (primeiras 2)
    if diffs['additions']:
        print(f"\nAdições ({len(diffs['additions'])} total):\n")
        for i, add in enumerate(diffs['additions'][:2], 1):
            print(f"{i}. Adicionado: '{add['content'][:100]}...'")
    
    # Mostrar remoções (primeiras 2)
    if diffs['removals']:
        print(f"\nRemoções ({len(diffs['removals'])} total):\n")
        for i, rem in enumerate(diffs['removals'][:2], 1):
            print(f"{i}. Removido: '{rem['original'][:100]}...'")
    
    print(f"\n{'='*80}")

# Estatísticas gerais
print(f"\n📊 Estatísticas de Comparação Textual:\n")
print(f"  Seções comparadas: {len(comparison_results)}")
print(f"  Tempo total: {sum(r['time'] for r in comparison_results):.2f}s")
print(f"  Tempo médio por seção: {sum(r['time'] for r in comparison_results) / len(comparison_results):.3f}s")
print(f"  Total de diferenças: {sum(r['total_diffs'] for r in comparison_results)}")

print(f"\n{'='*80}")
print("✅ Diferenças textuais detectadas! Prontas para análise semântica.")

---

## 7. Análise Semântica com Agno Framework

Esta seção **valida o objetivo primário do PoC**: integração do Agno Framework para classificação semântica de diferenças detectadas.

**Critério de sucesso**: Taxa de fallback <30% (Agno viável se ≥70% das chamadas funcionarem)

### 7.1 Criação do Agente Semântico

In [ ]:
# Importar módulo de análise semântica
from src.semantic_comparator import create_semantic_agent, classify_semantic_significance, get_semantic_stats

import time

# Criar agente Agno para análise semântica
print("🤖 Criando agente semântico com Agno Framework...")
start_time = time.time()

agent = create_semantic_agent()
elapsed = time.time() - start_time

print(f"✅ Agente criado com sucesso em {elapsed:.2f}s")
print(f"\nConfiguração:")
print(f"  Modelo: {MODEL}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Instruções: Especialista em análise de normas técnicas ASTM")

### 7.2 Teste Básico do Agente (Hello World)

In [ ]:
# Teste simples para validar que o agente está funcionando
print("🧪 Teste 1: Hello World com Agno")
print("-" * 60)

start_time = time.time()
response = agent.run("What is semantic equivalence in one sentence?")
elapsed = time.time() - start_time

print(f"✅ Resposta recebida em {elapsed:.2f}s:")
print(f"\n{response.content}\n")

### 7.3 Testes de Classificação Semântica

Vamos testar os 3 níveis de classificação: EQUIVALENT, MINOR, SIGNIFICANT

In [ ]:
# Definir casos de teste para cada nível de significância
test_cases = [
    {
        "name": "EQUIVALENT - Reformulação sem mudança de sentido",
        "diff": {"original": "automobile", "content": "vehicle"},
        "expected": "EQUIVALENT"
    },
    {
        "name": "MINOR - Clarificação editorial",
        "diff": {"original": "shall be tested", "content": "shall be tested for compliance"},
        "expected": "MINOR"
    },
    {
        "name": "SIGNIFICANT - Mudança de requisito crítico",
        "diff": {"original": "mandatory testing", "content": "optional testing"},
        "expected": "SIGNIFICANT"
    },
    {
        "name": "SIGNIFICANT - Mudança numérica crítica",
        "diff": {"original": "tensile strength ≥ 500 MPa", "content": "tensile strength ≥ 550 MPa"},
        "expected": "SIGNIFICANT"
    }
]

print("🧪 Testes de Classificação Semântica")
print("=" * 80)

results = []

for i, test in enumerate(test_cases, 1):
    print(f"\nTeste {i}: {test['name']}")
    print("-" * 80)
    print(f"Original: '{test['diff']['original']}'")
    print(f"Modificado: '{test['diff']['content']}'")
    print(f"Esperado: {test['expected']}")
    
    start_time = time.time()
    result = classify_semantic_significance(test['diff'], agent)
    elapsed = time.time() - start_time
    
    status = "✅" if result['classification'] == test['expected'] else "⚠️ "
    print(f"\n{status} Resultado: {result['classification']} (confiança: {result['confidence']:.2f})")
    print(f"Fonte: {result['source']}")
    print(f"Reasoning: {result['reasoning']}")
    print(f"Tempo: {elapsed:.2f}s")
    
    results.append({
        "test": test['name'],
        "expected": test['expected'],
        "got": result['classification'],
        "match": result['classification'] == test['expected'],
        "confidence": result['confidence'],
        "time": elapsed,
        "source": result['source']
    })

# Resumo
print("\n" + "=" * 80)
print("RESUMO DOS TESTES")
print("=" * 80)

correct = sum(1 for r in results if r['match'])
total = len(results)
accuracy = correct / total * 100

print(f"\nAcurácia: {correct}/{total} ({accuracy:.1f}%)")
print(f"Tempo médio: {sum(r['time'] for r in results) / total:.2f}s")

agno_calls = sum(1 for r in results if r['source'] == 'agno')
fallback_calls = sum(1 for r in results if r['source'] == 'openai_fallback')
print(f"\nChamadas Agno: {agno_calls}")
print(f"Fallbacks OpenAI: {fallback_calls}")

if fallback_calls / total > 0.3:
    print(f"\n⚠️  ATENÇÃO: Taxa de fallback ({fallback_calls/total*100:.1f}%) excede 30%!")
else:
    print(f"\n✅ Agno viável: {fallback_calls/total*100:.1f}% fallback (target: <30%)")

### 7.4 Estatísticas e Custo da Análise Semântica

In [ ]:
# Obter estatísticas da sessão de análise semântica
from src.semantic_comparator import log_semantic_summary

print("📊 Estatísticas da Análise Semântica")
print("=" * 80)

stats = get_semantic_stats()

print(f"\nChamadas LLM:")
print(f"  Total: {stats['total_llm_calls']}")
print(f"  Agno: {stats['agno_calls']} ({100 - stats['fallback_rate']:.1f}%)")
print(f"  OpenAI fallback: {stats['fallback_calls']} ({stats['fallback_rate']:.1f}%)")
print(f"  Cache hits: {stats['cache_hits']}")

print(f"\nCusto estimado: ${stats['total_cost']:.4f}")

print(f"\n{'='*80}")
print("VALIDAÇÃO DO PoC - VIABILIDADE DO AGNO FRAMEWORK")
print("="*80)

if stats['fallback_rate'] <= 30.0:
    print(f"\n✅ SUCESSO: Agno é VIÁVEL para produção")
    print(f"   Taxa de fallback: {stats['fallback_rate']:.1f}% (target: <30%)")
    print(f"   {100 - stats['fallback_rate']:.1f}% das chamadas funcionaram com Agno")
else:
    print(f"\n⚠️  ATENÇÃO: Taxa de fallback elevada")
    print(f"   Taxa de fallback: {stats['fallback_rate']:.1f}% (target: <30%)")
    print(f"   Considerar migração para OpenAI SDK direto")

# Log completo
print(f"\n{'='*80}")
log_semantic_summary()